# 🎙️ Benchmark : `bofenghuang/phonemizer-wav2vec2-ctc-french`
## Dataset : **Cnam-LMSSC/vibravox** — subset `speech_clean`, split `test`

---

### ✅ Pourquoi Vibravox est idéal pour ce benchmark

| Critère | Vibravox |
|---------|----------|
| Jamais vu pendant l'entraînement | ✅ (modèle entraîné sur Common Voice 13 seulement) |
| Parole réelle d'adultes natifs | ✅ 188 participants, hommes et femmes |
| Transcriptions IPA **déjà incluses** dans le dataset | ✅ colonne `phonemized_text` |
| Phrases et paragraphes variés | ✅ issues de Wikipedia français |
| Multi-capteurs (headset comme référence) | ✅ `audio.headset_microphone` = micro classique |
| Licence ouverte | ✅ CC-BY-4.0 |

### 🔑 Avantage clé
Vibravox contient déjà le champ **`phonemized_text`** : les références IPA sont **annotées manuellement** et corrigées par des experts — bien supérieur à une génération automatique par eSpeak.

### Structure du dataset
```
Cnam-LMSSC/vibravox
├── subset: speech_clean   ← environnement calme  (utilisé ici)
├── subset: speech_noisy   ← environnement bruité
├── subset: speechless_clean
└── subset: speechless_noisy

Capteurs audio disponibles :
  audio.headset_microphone       ← micro de référence (airborne)
  audio.throat_microphone        ← microphone laryngé
  audio.soft_in_ear_microphone   ← micro intra-auriculaire souple
  audio.rigid_in_ear_microphone  ← micro intra-auriculaire rigide
  audio.forehead_accelerometer   ← accéléromètre frontal
  audio.temple_vibration_pickup  ← capteur temporal
```

### Métriques calculées
| Métrique | Description |
|----------|-------------|
| **PER** | Phoneme Error Rate — métrique principale |
| **Substitutions / Insertions / Suppressions** | Détail des erreurs |
| **PER par phonème** | Quels phonèmes IPA posent problème |
| **PER par genre** | Hommes vs femmes |
| **PER par capteur** | Headset vs gorge vs intra |
| **RTF** | Real-Time Factor (vitesse d'inférence) |

## ℹ️ Mode local — dataset depuis `/kaggle/input/`

Ce notebook lit le fichier parquet **directement depuis l'input Kaggle** :
```
/kaggle/input/dataset/test-00000-of-00030.parquet
```
**Zéro téléchargement pendant l'exécution** — l'inférence démarre immédiatement après le chargement du modèle.

> Si tu as uploadé un autre fichier (ex: `test-00005-of-00030.parquet`), modifie `PARQUET_PATH` dans la cellule 4.


## 1. Installation des dépendances

In [ ]:
%%capture
# ── Fix prioritaire : compatibilité CUDA kernel ──────────────────────────────
# L'erreur "cudaErrorNoKernelImageForDevice" signifie que PyTorch a été compilé
# pour une architecture CUDA différente du GPU disponible sur cette session.
# Solution : réinstaller PyTorch avec le bon build CUDA pour le GPU actuel.

import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.returncode, result.stdout + result.stderr

# Détecter le GPU et sa compute capability
import torch
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU : {gpu_name}  (Compute Capability {cc[0]}.{cc[1]})")
else:
    print("Pas de GPU CUDA disponible")

# Vérifier la version CUDA runtime
code, out = run("nvcc --version 2>/dev/null || nvidia-smi | grep CUDA")
print(f"CUDA : {out.strip()[:80]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA build dans PyTorch : {torch.version.cuda}")

!pip install -q transformers datasets soundfile librosa jiwer torchaudio --quiet
!pip install -q "torch>=2.1" torchaudio --index-url https://download.pytorch.org/whl/cu121 --quiet


## 2. Imports et configuration

In [ ]:
import torch
import torchaudio
import numpy as np
import pandas as pd
import time
import warnings
import collections
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm
from transformers import AutoModelForCTC, AutoProcessor, pipeline
import jiwer

warnings.filterwarnings('ignore')

# ── Diagnostic GPU complet ───────────────────────────────────────────────────
print("=" * 55)
print("  DIAGNOSTIC GPU")
print("=" * 55)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    cc = torch.cuda.get_device_capability()
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU       : {gpu_name}")
    print(f"  Compute   : {cc[0]}.{cc[1]}")
    print(f"  VRAM      : {vram:.1f} GB")
    print(f"  PyTorch   : {torch.__version__}")
    print(f"  CUDA build: {torch.version.cuda}")
    
    # Tester si le GPU fonctionne réellement avec une opération simple
    try:
        t = torch.tensor([1.0, 2.0]).cuda()
        _ = t * 2
        del t
        print(f"  Test CUDA : OK ✅")
        DEVICE = "cuda"
    except Exception as e:
        print(f"  Test CUDA : ÉCHEC ❌ — {e}")
        print(f"  → Basculement sur CPU")
        DEVICE = "cpu"
else:
    print("  Pas de GPU CUDA — utilisation CPU")
    DEVICE = "cpu"

print("=" * 55)

# ── Configuration ────────────────────────────────────────────────────────────
MODEL_NAME  = "bofenghuang/phonemizer-wav2vec2-ctc-french"
SENSOR      = "audio.headset_microphone"
MAX_SAMPLES = None
TARGET_SR   = 16_000
FORCE_FP32  = True

ALL_SENSORS = [
    "audio.headset_microphone",
    "audio.throat_microphone",
    "audio.soft_in_ear_microphone",
    "audio.rigid_in_ear_microphone",
    "audio.forehead_accelerometer",
    "audio.temple_vibration_pickup",
]

print(f"\n📦 Modèle  : {MODEL_NAME}")
print(f"🎙️  Capteur : {SENSOR}")
print(f"🖥️  Device  : {DEVICE}")
print(f"🔢 Samples : {MAX_SAMPLES if MAX_SAMPLES else 'tous'}")


## 3. Chargement du modèle

In [ ]:
print(f"⏳ Chargement de {MODEL_NAME} ...")
t0 = time.time()

# Float32 obligatoire — float16 cause AcceleratorError sur GPU T4/V100
torch_dtype = torch.float32

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model     = AutoModelForCTC.from_pretrained(MODEL_NAME, torch_dtype=torch_dtype)

# ── Chargement robuste avec fallback CPU si CUDA échoue ────────────────────
def load_model_safe(model, device):
    """Tente de charger sur GPU, bascule sur CPU si erreur CUDA kernel."""
    if device == "cpu":
        model = model.to("cpu")
        return model, "cpu"
    try:
        model = model.to(device)
        # Test réel : une petite inférence pour détecter cudaErrorNoKernelImageForDevice
        dummy = torch.zeros(1, 16000, dtype=torch_dtype).to(device)
        with torch.no_grad():
            _ = model.wav2vec2.feature_extractor(dummy)
        del dummy
        return model, device
    except Exception as e:
        print(f"\n⚠️  GPU indisponible : {str(e)[:120]}")
        print(f"   → Basculement automatique sur CPU")
        model = model.to("cpu")
        return model, "cpu"

model, DEVICE = load_model_safe(model, DEVICE)
model.eval()

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    feature_extractor=processor.feature_extractor,
    tokenizer=processor.tokenizer,
    device=DEVICE,
)

elapsed = time.time() - t0
print(f"\n✅ Modèle chargé en {elapsed:.1f}s")
print(f"   dtype   : float32")
print(f"   device  : {DEVICE}")
print(f"   params  : {sum(p.numel() for p in model.parameters())/1e6:.0f}M")
print(f"   vocab   : {processor.tokenizer.vocab_size} tokens IPA")

if DEVICE == "cpu":
    print(f"\n⚠️  Mode CPU actif — inférence ~10× plus lente")
    print(f"   Raison possible : GPU incompatible avec ce build PyTorch")
    print(f"   Solution : relancer la session Kaggle pour obtenir un autre GPU")
    print(f"   Ou activer l'accélérateur P100/T4 dans Paramètres → GPU")


## 4. Chargement du dataset Vibravox

**Point fort** : Vibravox contient directement la colonne `phonemized_text` —  
les références IPA sont annotées et vérifiées manuellement. Pas besoin de G2P logiciel.

In [ ]:
import os, glob

# ── Trouver automatiquement le fichier parquet ────────────────────────────────
print("📂 Contenu de /kaggle/input/ :")
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        size_mb = os.path.getsize(os.path.join(root, f)) / 1e6
        print(f"{indent}  {f}  ({size_mb:.0f} MB)")

# Chercher automatiquement tous les .parquet
parquets = glob.glob("/kaggle/input/**/*.parquet", recursive=True)
print(f"\n✅ Fichiers .parquet trouvés :")
for p in parquets:
    print(f"   {p}")

# Sélectionner le premier automatiquement
if parquets:
    PARQUET_PATH = parquets[0]
    print(f"\n🎯 PARQUET_PATH = '{PARQUET_PATH}'")
else:
    print("\n❌ Aucun fichier .parquet trouvé dans /kaggle/input/")
    print("   Vérifie que le dataset est bien ajouté en Input dans le notebook.")


In [ ]:
import pandas as pd
import numpy as np
import io, glob, os
import soundfile as sf

# ── Détection séparée clean / noisy ─────────────────────────────────────────
all_parquets   = sorted(glob.glob("/kaggle/input/**/*.parquet", recursive=True))
clean_parquets = [p for p in all_parquets if "noise" not in p.lower()]
noisy_parquets = [p for p in all_parquets if "noise"     in p.lower()]

print(f"📂 Parquets détectés :")
print(f"   speech_clean  ({len(clean_parquets)} fichiers) :")
for p in clean_parquets:
    print(f"     {os.path.basename(p)}  ({os.path.getsize(p)//1_000_000} MB)")
print(f"   speech_noisy  ({len(noisy_parquets)} fichiers) :")
for p in noisy_parquets:
    print(f"     {os.path.basename(p)}  ({os.path.getsize(p)//1_000_000} MB)")

if not clean_parquets:
    raise FileNotFoundError("Aucun parquet clean trouvé.")

# ── Chargement speech_clean uniquement pour df_raw ───────────────────────────
print(f"\n⏳ Chargement speech_clean ({len(clean_parquets)} fichiers) ...")
dfs = []
for p in clean_parquets:
    part = pd.read_parquet(p)
    dfs.append(part)
    print(f"   {os.path.basename(p)} → {len(part)} lignes")

df_raw = pd.concat(dfs, ignore_index=True)
df_raw = df_raw.drop_duplicates(subset=["sentence_id", "speaker_id"]).reset_index(drop=True)

if MAX_SAMPLES is not None:
    df_raw = df_raw.head(MAX_SAMPLES).reset_index(drop=True)

male_n   = (df_raw["gender"] == "male").sum()
female_n = (df_raw["gender"] == "female").sum()
print(f"\n✅ df_raw (speech_clean) : {len(df_raw)} exemples")
print(f"   Hommes   : {male_n} ({male_n/len(df_raw)*100:.0f}%)")
print(f"   Femmes   : {female_n} ({female_n/len(df_raw)*100:.0f}%)")
print(f"   Locuteurs uniques : {df_raw['speaker_id'].nunique()}")
print(f"   Durée totale audio : {df_raw['duration'].sum()/60:.1f} min")
print(f"   Colonnes : {list(df_raw.columns)}")

row = df_raw.iloc[0]
print(f"\n📝 Exemple #0 :")
print(f"   raw_text        : {row['raw_text']}")
print(f"   phonemized_text : {row['phonemized_text']}")
print(f"   gender          : {row['gender']}")
print(f"   duration        : {row['duration']:.2f}s")


## 5. Inférence — prédiction IPA depuis l'audio

On utilise le capteur `headset_microphone` (micro casque airborne standard).  
L'audio Vibravox est à **48 kHz** → rééchantillonnage à **16 kHz** requis par le modèle.

In [ ]:
import io, json as _json, os as _os
import soundfile as sf

# Reprise depuis checkpoint si disponible, sinon inférence complète
CHECKPOINT = "/kaggle/working/results_checkpoint.json"

def resample_to_16k(arr: np.ndarray, orig_sr: int) -> np.ndarray:
    if orig_sr == TARGET_SR:
        return arr
    import torch, torchaudio
    w = torchaudio.functional.resample(
        torch.tensor(arr, dtype=torch.float32).unsqueeze(0),
        orig_freq=orig_sr, new_freq=TARGET_SR
    )
    return w.squeeze(0).numpy()

def extract_audio(audio_field) -> tuple:
    """Extrait (array_float32, sample_rate) depuis le champ audio du parquet Vibravox."""
    if isinstance(audio_field, dict):
        raw = audio_field.get('bytes') or audio_field.get('array')
        if isinstance(raw, bytes):
            arr, sr = sf.read(io.BytesIO(raw))
            return arr.astype(np.float32), sr
        if hasattr(raw, '__len__'):
            sr = audio_field.get('sampling_rate', 48_000)
            return np.array(raw, dtype=np.float32), sr
    if isinstance(audio_field, bytes):
        arr, sr = sf.read(io.BytesIO(audio_field))
        return arr.astype(np.float32), sr
    raise ValueError(f"Format audio non reconnu : {type(audio_field)}")

# Reprendre depuis checkpoint si existant ET si même taille de dataset
results    = []
start_idx  = 0
if _os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        saved = _json.load(f)
    if len(saved) < len(df_raw):
        results   = saved
        start_idx = len(saved)
        print(f"♻️  Checkpoint : {start_idx}/{len(df_raw)} déjà traités, reprise...")
    else:
        print(f"♻️  Checkpoint complet ({len(saved)} ex.) — rechargement direct")
        results = saved
        start_idx = len(saved)

total_audio_s = sum(r["duration"]       for r in results)
total_infer_s = sum(r["inference_time"] for r in results)

if start_idx < len(df_raw):
    print(f"🚀 Inférence sur {len(df_raw) - start_idx} exemples restants...\n")

    for idx, row in tqdm(df_raw.iterrows(), total=len(df_raw), desc="Inférence"):
        if idx < start_idx:
            continue

        ref_ipa  = str(row["phonemized_text"]).strip()
        raw_text = str(row["raw_text"])
        gender   = str(row["gender"])
        speaker  = str(row["speaker_id"])
        duration = float(row["duration"])

        try:
            audio_arr, orig_sr = extract_audio(row[SENSOR])
            audio_16k = resample_to_16k(audio_arr, orig_sr)
        except Exception as e:
            print(f"⚠️  Erreur audio idx={idx}: {e}")
            continue

        t_start = time.time()
        with torch.inference_mode():
            pred = asr_pipe({"array": audio_16k, "sampling_rate": TARGET_SR})
        t_infer = time.time() - t_start

        pred_ipa = pred["text"].strip()
        total_audio_s += duration
        total_infer_s += t_infer

        results.append({
            "idx":            int(idx),
            "raw_text":       raw_text,
            "ref_ipa":        ref_ipa,
            "pred_ipa":       pred_ipa,
            "gender":         gender,
            "speaker_id":     speaker,
            "duration":       duration,
            "inference_time": t_infer,
        })

        if len(results) % 50 == 0:
            with open(CHECKPOINT, "w") as f:
                _json.dump(results, f)

    with open(CHECKPOINT, "w") as f:
        _json.dump(results, f)

rtf = total_infer_s / max(total_audio_s, 1)
print(f"\n✅ {len(results)} inférences terminées")
print(f"   Durée audio    : {total_audio_s/60:.1f} min")
print(f"   Temps inférence: {total_infer_s:.1f}s")
print(f"   RTF            : {rtf:.4f}  ({1/max(rtf,0.001):.1f}x temps réel)")


## 6. Calcul du Phoneme Error Rate (PER)

Vibravox fournit les phonèmes IPA déjà sous forme de séquence de caractères séparés par des espaces (un espace entre chaque mot phonémisé).  
On traite chaque **caractère IPA** comme un token pour calculer le PER.

In [ ]:
# Déduplication résultats — clé robuste sur sentence_id
seen = set()
results_dedup = []
for r in results:
    # FIX C2 : utiliser sentence_id (identifiant officiel Vibravox) au lieu de ref_ipa[:20]
    sid = r.get("sentence_id") or r.get("idx")
    if sid not in seen:
        seen.add(sid)
        results_dedup.append(r)
results = results_dedup
print(f"✅ {len(results)} résultats après déduplication")

def per_from_ipa(reference: str, hypothesis: str) -> dict:
    """Calcule le PER en tokenisant chaque caractère IPA. Compatible jiwer >= 3.0."""
    if not reference or not hypothesis:
        return {"per": 1.0, "substitutions": 0, "insertions": 0, "deletions": 0, "hits": 0}
    ref_tokens = " ".join(list(reference.replace(" ", "_")))
    hyp_tokens = " ".join(list(hypothesis.replace(" ", "_")))
    out = jiwer.process_words(ref_tokens, hyp_tokens)
    return {
        "per":           out.wer,
        "substitutions": out.substitutions,
        "insertions":    out.insertions,
        "deletions":     out.deletions,
        "hits":          out.hits,
    }

# ── Calcul PER pour chaque exemple ──────────────────────────────────────────
per_list = []
total_sub = total_ins = total_del = total_hits = 0

for r in results:
    m = per_from_ipa(r["ref_ipa"], r["pred_ipa"])
    r.update(m)
    per_list.append(m["per"])
    total_sub  += m["substitutions"]
    total_ins  += m["insertions"]
    total_del  += m["deletions"]
    total_hits += m["hits"]

denom      = total_sub + total_del + total_hits
global_per = (total_sub + total_ins + total_del) / max(denom, 1)
rtf        = total_infer_s / max(total_audio_s, 1)

# ── Intervalle de confiance 95% par bootstrap (E1) ──────────────────────────
import numpy as np
rng = np.random.default_rng(42)
n_boot = 2000
boot_means = [
    np.mean(rng.choice(per_list, size=len(per_list), replace=True))
    for _ in range(n_boot)
]
ci_lo = np.percentile(boot_means, 2.5) * 100
ci_hi = np.percentile(boot_means, 97.5) * 100

print("=" * 60)
print(f"  BENCHMARK — {MODEL_NAME}")
print(f"  Dataset : Vibravox speech_clean / test / {len(results)} ex.")
print(f"  Capteur : {SENSOR}")
print("=" * 60)
print(f"  PER global              : {global_per*100:>7.2f} %")
print(f"  IC 95% bootstrap        :  [{ci_lo:.2f}% — {ci_hi:.2f}%]")
print(f"  PER médian (par ex.)    : {np.median(per_list)*100:>7.2f} %")
print(f"  PER moyen  (par ex.)    : {np.mean(per_list)*100:>7.2f} %")
print(f"  PER std                 : {np.std(per_list)*100:>7.2f} %")
print(f"  PER min                 : {np.min(per_list)*100:>7.2f} %")
print(f"  PER max                 : {np.max(per_list)*100:>7.2f} %")
print("-" * 60)
print(f"  Substitutions           : {total_sub:>8d}")
print(f"  Insertions              : {total_ins:>8d}")
print(f"  Suppressions            : {total_del:>8d}")
print(f"  Correct (hits)          : {total_hits:>8d}")
print("-" * 60)
print(f"  RTF moyen               : {rtf:.4f}x  ({1/max(rtf,0.001):.1f}x temps réel)")
print("=" * 60)


## 7. Analyse par genre (Hommes vs Femmes)

In [ ]:
from scipy import stats

df = pd.DataFrame(results)

# Calcul PER par genre
gender_stats = (
    df.groupby("gender")["per"]
    .agg(count="count",
         per_mean=lambda x: x.mean() * 100,
         per_median=lambda x: x.median() * 100,
         per_std=lambda x: x.std() * 100)
    .reset_index()
)

print("\n📊 PER par genre :")
print(gender_stats.to_string(index=False))
for _, row in gender_stats.iterrows():
    symbol = "🔵" if row["gender"] == "male" else "🔴"
    bar    = "█" * int(row["per_mean"] / 2)
    print(f"  {symbol} {row['gender']:>6} : {row['per_mean']:>5.1f}%  {bar}")

# ── Test de Wilcoxon (C4) ────────────────────────────────────────────────────
male_per   = df[df["gender"]=="male"]["per"].values
female_per = df[df["gender"]=="female"]["per"].values

# Wilcoxon rank-sum (Mann-Whitney) — indépendant, échantillons de tailles différentes
stat, pval = stats.mannwhitneyu(male_per, female_per, alternative="two-sided")
sig = "✅ significatif (p<0.05)" if pval < 0.05 else "⚠️  non significatif (p≥0.05)"
print(f"\n  Test Mann-Whitney H/F : p={pval:.4f} → {sig}")
print(f"  Interprétation : {'la différence de PER H/F est réelle' if pval < 0.05 else 'la différence de PER H/F peut être due au hasard'}")

# ── Analyse par locuteur (E2) ────────────────────────────────────────────────
print("\n📊 PER par locuteur (speaker_id) :")
speaker_stats = (
    df.groupby(["speaker_id","gender"])["per"]
    .agg(n="count", per_mean=lambda x: x.mean()*100, per_std=lambda x: x.std()*100)
    .sort_values("per_mean", ascending=False)
    .reset_index()
)
print(f"  {'Speaker':<14} {'G':>2} {'N':>4} {'PER moy':>8} {'PER std':>8}")
print("  " + "-"*42)
for _, row in speaker_stats.iterrows():
    bar = "█" * int(row["per_mean"] / 1.5)
    g   = "M" if row["gender"]=="male" else "F"
    print(f"  {row['speaker_id']:<14} {g:>2} {int(row['n']):>4} {row['per_mean']:>7.2f}%  {row['per_std']:>7.2f}%  {bar}")

print(f"\n  Variance inter-locuteurs : std={speaker_stats['per_mean'].std():.2f}%")
print(f"  Locuteur le pire  : {speaker_stats.iloc[0]['speaker_id']} ({speaker_stats.iloc[0]['per_mean']:.2f}%)")
print(f"  Locuteur le meilleur : {speaker_stats.iloc[-1]['speaker_id']} ({speaker_stats.iloc[-1]['per_mean']:.2f}%)")


## 8. Exemples qualitatifs — Meilleurs et pires prédictions

In [ ]:
df_sorted = df.sort_values("per")

print("\n🏆 TOP 5 — Meilleures prédictions (PER le plus bas)")
print("=" * 85)
for _, row in df_sorted.head(5).iterrows():
    print(f"  PER: {row['per']*100:.1f}%  |  Genre: {row['gender']}  |  Durée: {row['duration']:.2f}s")
    print(f"  📝 Texte  : {row['raw_text']}")
    print(f"  🔵 Réf IPA: {row['ref_ipa']}")
    print(f"  🟢 Préd   : {row['pred_ipa']}")
    print()

print("\n❌ TOP 5 — Pires prédictions (PER le plus haut)")
print("=" * 85)
for _, row in df_sorted.tail(5).iterrows():
    print(f"  PER: {row['per']*100:.1f}%  |  Genre: {row['gender']}  |  Durée: {row['duration']:.2f}s")
    print(f"  📝 Texte  : {row['raw_text']}")
    print(f"  🔵 Réf IPA: {row['ref_ipa']}")
    print(f"  🔴 Préd   : {row['pred_ipa']}")
    print()

## 9. Analyse par phonème IPA — quels phonèmes posent le plus de problèmes ?

Vibravox utilise un jeu strict de phonèmes IPA français :  
`a b d e f i j k l m n o p s t u v w y z ø ŋ œ ɑ ɔ ə ɛ ɡ ɲ ʁ ʃ ʒ ̃`

In [ ]:
def get_errored_phones(ref_str: str, hyp_str: str) -> list:
    """Retourne les phonèmes de référence substitués ou supprimés (jiwer >= 3.0)."""
    ref_tok = list(ref_str.replace(" ", "_"))
    hyp_tok = list(hyp_str.replace(" ", "_"))
    out     = jiwer.process_words(" ".join(ref_tok), " ".join(hyp_tok))
    errors  = []
    for chunk in out.alignments[0]:
        if chunk.type in ("substitution", "delete"):
            for i in range(chunk.ref_start_idx, chunk.ref_end_idx):
                if i < len(ref_tok) and ref_tok[i] != "_":
                    errors.append(ref_tok[i])
    return errors


phone_errors = collections.Counter()
phone_total  = collections.Counter()

sample_for_analysis = results  # FIX C2 : analyse sur TOUS les exemples (plus min(150))

for r in tqdm(sample_for_analysis, desc="Analyse phonèmes"):
    if not r.get("ref_ipa") or not r.get("pred_ipa"):
        continue
    for p in r["ref_ipa"].replace(" ", ""):
        phone_total[p] += 1
    for p in get_errored_phones(r["ref_ipa"], r["pred_ipa"]):
        phone_errors[p] += 1

phone_per_dict = {}
for ph, err in phone_errors.most_common():
    if phone_total[ph] >= 5:
        phone_per_dict[ph] = err / phone_total[ph]

top_hard = sorted(phone_per_dict.items(), key=lambda x: -x[1])[:20]

print(f"\n📊 Top 20 phonèmes les plus difficiles (sur tous les exemples) :")
print(f"  {'IPA':>6}  {'Taux err.':>10}  {'Erreurs':>8}  {'Total':>8}")
print("-" * 42)
for ph, rate in top_hard:
    icon = "🔴" if rate > 0.4 else "🟠" if rate > 0.2 else "🟡"
    print(f"  {icon} {ph:>4}   {rate*100:>8.1f}%   {phone_errors[ph]:>6}    {phone_total[ph]:>6}")


## 10. Analyse multi-capteurs (optionnel)

Vibravox contient 6 capteurs. On teste ici le modèle sur chaque capteur  
pour voir si les corps-conduction dégradent les performances de phonémisation.

> ⚠️ Cette cellule est **optionnelle** — elle lance 5 inférences supplémentaires.  
> Mettre `RUN_MULTI_SENSOR = True` pour l'activer.

In [ ]:
RUN_MULTI_SENSOR = False   # True = compare les 6 capteurs (nécessite toutes les colonnes audio)
N_SAMPLES_SENSOR = 50

if RUN_MULTI_SENSOR:
    sensor_results = {}
    dataset_sub = df_raw.head(min(N_SAMPLES_SENSOR, len(df_raw)))

    for sensor in ALL_SENSORS:
        per_sensor = []
        for _, row in tqdm(dataset_sub.iterrows(), total=len(dataset_sub),
                           desc=f"Capteur: {sensor.split('.')[-1]}", leave=False):
            ref_ipa = str(row["phonemized_text"]).strip()
            try:
                audio_arr, orig_sr = extract_audio(row[sensor])
                audio_16k = resample_to_16k(audio_arr, orig_sr)
            except Exception:
                continue
            with torch.inference_mode():
                pred = asr_pipe({"array": audio_16k, "sampling_rate": TARGET_SR})
            pred_ipa = pred["text"].strip()
            m = per_from_ipa(ref_ipa, pred_ipa)
            per_sensor.append(m["per"])

        sensor_results[sensor] = {
            "mean_per":   np.mean(per_sensor) * 100,
            "median_per": np.median(per_sensor) * 100,
        }

    print("\n📊 PER par capteur :")
    for s, v in sorted(sensor_results.items(), key=lambda x: x[1]["mean_per"]):
        ref = " ← référence" if s == SENSOR else ""
        print(f"  {s.replace('audio.',''):<30} : {v['mean_per']:>6.1f}%{ref}")
else:
    sensor_results = None
    print("ℹ️  Multi-capteurs désactivé (RUN_MULTI_SENSOR = False)")


## 11. Visualisations complètes

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    f"Benchmark : {MODEL_NAME}\nVibravox speech_clean / {SENSOR} — {len(results)} exemples",
    fontsize=13, fontweight='bold', y=1.01
)

# ── 1. Distribution PER ─────────────────────────────────────────────────────
ax1 = fig.add_subplot(3, 3, 1)
ax1.hist(np.array(per_list)*100, bins=30, color="#4C72B0", edgecolor="white", alpha=0.85)
ax1.axvline(np.mean(per_list)*100,   color="red",    linestyle="--", linewidth=1.5, label=f"Moy: {np.mean(per_list)*100:.1f}%")
ax1.axvline(np.median(per_list)*100, color="orange", linestyle=":",  linewidth=1.5, label=f"Méd: {np.median(per_list)*100:.1f}%")
ax1.set_xlabel("PER (%)")
ax1.set_ylabel("Exemples")
ax1.set_title("Distribution du PER")
ax1.legend(fontsize=8); ax1.grid(axis='y', alpha=0.3)

# ── 2. Erreurs : sub / ins / del ────────────────────────────────────────────
ax2 = fig.add_subplot(3, 3, 2)
vals   = [total_sub, total_ins, total_del]
labels = ["Substitutions", "Insertions", "Suppressions"]
cols   = ["#e07b54", "#5b8dd9", "#56b356"]
bars = ax2.bar(labels, vals, color=cols, edgecolor="white", alpha=0.9)
for b, v in zip(bars, vals):
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.01, str(v),
             ha='center', va='bottom', fontweight='bold', fontsize=10)
ax2.set_title("Types d'erreurs (total)")
ax2.set_ylabel("Nombre")
ax2.grid(axis='y', alpha=0.3)

# ── 3. PER par genre ────────────────────────────────────────────────────────
ax3 = fig.add_subplot(3, 3, 3)
male_per   = [r["per"]*100 for r in results if r["gender"] == "male"]
female_per = [r["per"]*100 for r in results if r["gender"] == "female"]
ax3.boxplot([male_per, female_per], labels=["Homme", "Femme"],
            patch_artist=True,
            boxprops=dict(facecolor="#5b8dd9", alpha=0.7),
            medianprops=dict(color="red", linewidth=2))
ax3.set_ylabel("PER (%)")
ax3.set_title("PER par genre")
ax3.grid(axis='y', alpha=0.3)

# ── 4. Top 15 phonèmes difficiles ────────────────────────────────────────────
ax4 = fig.add_subplot(3, 3, 4)
if top_hard:
    phones_h = [p for p, _ in top_hard[:15]]
    rates_h  = [r*100 for _, r in top_hard[:15]]
    cols_h   = ["#d62728" if r > 40 else "#ff7f0e" if r > 20 else "#2ca02c" for r in rates_h]
    ax4.barh(phones_h[::-1], rates_h[::-1], color=cols_h[::-1], edgecolor="white", alpha=0.9)
    ax4.axvline(global_per*100, color="black", linestyle="--", linewidth=1, alpha=0.6,
                label=f"PER global: {global_per*100:.1f}%")
    ax4.set_xlabel("Taux d'erreur (%)")
    ax4.set_title("Top 15 phonèmes difficiles")
    ax4.legend(fontsize=8); ax4.grid(axis='x', alpha=0.3)

# ── 5. PER vs durée de l'audio ───────────────────────────────────────────────
ax5 = fig.add_subplot(3, 3, 5)
durs = [r["duration"]   for r in results if "per" in r]
pers = [r["per"]*100    for r in results if "per" in r]
gend = [r["gender"]     for r in results if "per" in r]
colors_scatter = ["#5b8dd9" if g == "male" else "#e07b54" for g in gend]
ax5.scatter(durs, pers, alpha=0.3, s=15, c=colors_scatter, edgecolors="none")
z = np.polyfit(durs, pers, 1); p = np.poly1d(z)
xs = sorted(durs)
ax5.plot(xs, p(xs), "k--", linewidth=1.5, alpha=0.6, label="Tendance")
blue_p = mpatches.Patch(color='#5b8dd9', label='Homme')
red_p  = mpatches.Patch(color='#e07b54', label='Femme')
ax5.legend(handles=[blue_p, red_p], fontsize=8)
ax5.set_xlabel("Durée (s)"); ax5.set_ylabel("PER (%)")
ax5.set_title("PER vs durée"); ax5.grid(alpha=0.3)

# ── 6. Fréquence des phonèmes de référence ──────────────────────────────────
ax6 = fig.add_subplot(3, 3, 6)
top_freq = phone_total.most_common(20)
phs_freq = [p for p, _ in top_freq]
cnt_freq = [c for _, c in top_freq]
ax6.bar(phs_freq, cnt_freq, color="#9467bd", edgecolor="white", alpha=0.85)
ax6.set_title("Fréquence des phonèmes (réf.)")
ax6.set_ylabel("Occurrences")
ax6.set_xlabel("Phonème IPA")
ax6.tick_params(axis='x', labelsize=9)
ax6.grid(axis='y', alpha=0.3)

# ── 7. Multi-capteurs (si disponible) ───────────────────────────────────────
ax7 = fig.add_subplot(3, 3, (7, 9))
if sensor_results:
    sorted_sensors = sorted(sensor_results.items(), key=lambda x: x[1]["mean_per"])
    s_names  = [s.replace("audio.", "").replace("_", "\n") for s, _ in sorted_sensors]
    s_means  = [v["mean_per"] for _, v in sorted_sensors]
    s_medians= [v["median_per"] for _, v in sorted_sensors]
    x_pos = np.arange(len(s_names))
    w = 0.35
    b1 = ax7.bar(x_pos - w/2, s_means,   w, label="PER moyen",  color="#4C72B0", alpha=0.85)
    b2 = ax7.bar(x_pos + w/2, s_medians, w, label="PER médian", color="#DD8452", alpha=0.85)
    for b, v in zip(b1, s_means):
        ax7.text(b.get_x()+b.get_width()/2, b.get_height()+0.2, f"{v:.1f}%", ha='center', fontsize=8, fontweight='bold')
    for b, v in zip(b2, s_medians):
        ax7.text(b.get_x()+b.get_width()/2, b.get_height()+0.2, f"{v:.1f}%", ha='center', fontsize=8, fontweight='bold')
    ax7.set_xticks(x_pos); ax7.set_xticklabels(s_names, fontsize=9)
    ax7.set_ylabel("PER (%)")
    ax7.set_title(f"Comparaison PER par capteur (sur {N_SAMPLES_SENSOR} ex.)")
    ax7.legend(fontsize=9); ax7.grid(axis='y', alpha=0.3)
else:
    ax7.text(0.5, 0.5, "Analyse multi-capteurs désactivée\n(RUN_MULTI_SENSOR = False)",
             ha='center', va='center', fontsize=12, color='gray', transform=ax7.transAxes)
    ax7.set_title("Comparaison par capteur")

plt.tight_layout()
plt.savefig("/kaggle/working/vibravox_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n📈 Graphiques sauvegardés : vibravox_benchmark.png")

## 12. PER par catégorie phonétique

In [ ]:
# Vocabulaire Vibravox strict (depuis la doc officielle)
VIBRAVOX_CATEGORIES = {
    "Voyelles orales":  list("aeiouɛɔœøy"),
    "Voyelles nasales": ["ɑ̃", "ɔ̃", "ɛ̃", "œ̃"],   # représentées par ɑ+̃, etc.
    "Schwa":            ["ə"],
    "Occlusives":       list("pbtkdɡ"),
    "Fricatives":       list("fvszʒʃ"),
    "Nasales":          list("mnɲŋ"),
    "Liquides":         ["l", "ʁ"],
    "Glides":           ["j", "w"],
}

print("\n📊 PER par catégorie phonétique (Vibravox) :")
print(f"{'Catégorie':<22}  {'PER':>7}  {'Erreurs':>8}  {'Total':>8}")
print("-" * 55)

cat_for_plot = {}
for cat, phones in VIBRAVOX_CATEGORIES.items():
    cat_err   = sum(phone_errors.get(p, 0) for p in phones)
    cat_tot   = sum(phone_total.get(p, 0)  for p in phones)
    if cat_tot > 0:
        rate = cat_err / cat_tot * 100
        cat_for_plot[cat] = rate
        icon = "🔴" if rate > 30 else "🟠" if rate > 15 else "🟢"
        bar  = "█" * int(rate / 3)
        print(f"  {icon} {cat:<20}  {rate:>6.1f}%  {cat_err:>8}  {cat_tot:>8}  {bar}")

# Graphique
fig2, ax = plt.subplots(figsize=(11, 5))
cats_plot = list(cat_for_plot.keys())
vals_plot = list(cat_for_plot.values())
cols_plot = ["#d62728" if v > 30 else "#ff7f0e" if v > 15 else "#2ca02c" for v in vals_plot]
bars = ax.bar(cats_plot, vals_plot, color=cols_plot, edgecolor="white", alpha=0.9)
for b, v in zip(bars, vals_plot):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
            f"{v:.1f}%", ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.axhline(global_per*100, color="black", linestyle="--", linewidth=1.2,
           alpha=0.7, label=f"PER global: {global_per*100:.1f}%")
ax.set_title("PER par catégorie phonétique — Vibravox (headset_microphone)", fontweight='bold')
ax.set_ylabel("Phoneme Error Rate (%)")
ax.set_ylim(0, max(vals_plot)*1.25 if vals_plot else 50)
plt.xticks(rotation=20, ha='right')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/vibravox_categories.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n📈 Graphique sauvegardé : vibravox_categories.png")

## 13b. Analyse PER par longueur de phrase

Vérifie si les phrases longues ont plus d'erreurs (frontières de mots, disfluences).

In [ ]:
# ── PER vs longueur de phrase (E5) ───────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats as _stats

df_len = pd.DataFrame(results).copy()
df_len["n_phonemes"] = df_len["ref_ipa"].apply(
    lambda x: len([c for c in x.replace(" ","") if c.strip()])
)
df_len["per_pct"] = df_len["per"] * 100

# Corrélation Spearman
rho, p_rho = _stats.spearmanr(df_len["n_phonemes"], df_len["per_pct"])
print(f"Corrélation Spearman PER ~ longueur : ρ={rho:.3f}  p={p_rho:.4f}")
print(f"{'→ Corrélation significative' if p_rho < 0.05 else '→ Pas de corrélation significative'} entre longueur et taux d'erreur")

# Tranches de longueur
df_len["tranche"] = pd.cut(df_len["n_phonemes"], bins=[0,20,35,50,200],
                           labels=["<20 ph","20-35 ph","35-50 ph",">50 ph"])
tranche_stats = df_len.groupby("tranche")["per_pct"].agg(
    n="count", moy="mean", med="median"
).reset_index()
print("\nPER par tranche de longueur :")
print(tranche_stats.to_string(index=False))

# Graphique scatter + tendance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(df_len["n_phonemes"], df_len["per_pct"],
                c=["#5b8dd9" if g=="male" else "#e07b54" for g in df_len["gender"]],
                alpha=0.5, s=25)
m, b = np.polyfit(df_len["n_phonemes"], df_len["per_pct"], 1)
x_line = np.linspace(df_len["n_phonemes"].min(), df_len["n_phonemes"].max(), 100)
axes[0].plot(x_line, m*x_line+b, "k--", linewidth=1.5, alpha=0.7,
             label=f"Tendance (ρ={rho:.2f})")
axes[0].set_xlabel("Nombre de phonèmes"); axes[0].set_ylabel("PER (%)")
axes[0].set_title("PER vs longueur de phrase"); axes[0].legend(); axes[0].grid(alpha=0.3)
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(color="#5b8dd9", label="Hommes"),
    Patch(color="#e07b54", label="Femmes"),
    plt.Line2D([],[],color="k",linestyle="--",label=f"Tendance ρ={rho:.2f}"),
])
axes[1].boxplot(
    [df_len[df_len["tranche"]==t]["per_pct"].values for t in tranche_stats["tranche"]],
    labels=tranche_stats["tranche"].tolist(), patch_artist=True,
    boxprops=dict(facecolor="#5b8dd9", alpha=0.7),
    medianprops=dict(color="red", linewidth=2)
)
axes[1].set_ylabel("PER (%)"); axes[1].set_title("Distribution PER par tranche")
axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("/kaggle/working/per_vs_length.png", dpi=150)
plt.show()
print("\n📈 Graphique sauvegardé : per_vs_length.png")


## 14. Export des résultats

In [ ]:
# CSV détaillé par exemple
df_export = df[[
    "idx", "raw_text", "ref_ipa", "pred_ipa",
    "per", "substitutions", "insertions", "deletions",
    "gender", "speaker_id", "duration", "inference_time"
]].copy()
df_export["per_pct"] = (df_export["per"] * 100).round(3)

csv_path = "/kaggle/working/vibravox_benchmark_details.csv"
df_export.to_csv(csv_path, index=False, encoding="utf-8")

# Résumé global
summary = {
    "modele":            MODEL_NAME,
    "dataset": "Cnam-LMSSC/vibravox/speech_clean",
    "split":   "test (parquet local)",
    "capteur":           SENSOR,
    "n_samples":         len(results),
    "per_global_pct":    round(global_per * 100, 3),
    "per_median_pct":    round(np.median(per_list) * 100, 3),
    "per_mean_pct":      round(np.mean(per_list) * 100, 3),
    "per_std_pct":       round(np.std(per_list) * 100, 3),
    "per_male_pct":      round(np.mean([r["per"] for r in results if r["gender"]=="male"])*100, 3) if male_per else None,
    "per_female_pct":    round(np.mean([r["per"] for r in results if r["gender"]=="female"])*100, 3) if female_per else None,
    "substitutions":     total_sub,
    "insertions":        total_ins,
    "deletions":         total_del,
    "rtf_mean":          round(rtf, 4),
    "device":            DEVICE,
}

summary_path = "/kaggle/working/vibravox_benchmark_summary.csv"
pd.DataFrame([summary]).to_csv(summary_path, index=False)

print("✅ Fichiers exportés :")
print(f"   📄 {csv_path}")
print(f"   📄 {summary_path}")
print(f"   📈 /kaggle/working/vibravox_benchmark.png")
print(f"   📈 /kaggle/working/vibravox_categories.png")
print(f"\n📌 Résumé :")
for k, v in summary.items():
    print(f"   {k:<26} : {v}")

## 15. Comparaison Clean vs Noisy

Lance l'inférence sur `speech_noisy` et compare directement avec les résultats `speech_clean` déjà calculés.

In [ ]:
import io, json as _json, os as _os, glob, time
import pandas as pd
import numpy as np
import soundfile as sf
import torch
from tqdm.auto import tqdm
import jiwer

# ── 1. Détecter les parquets noise vs clean ──────────────────────────────────
all_parquets = sorted(glob.glob("/kaggle/input/**/*.parquet", recursive=True))

noise_parquets = [p for p in all_parquets if "noise" in p.lower()]
clean_parquets = [p for p in all_parquets if "noise" not in p.lower()]

print("📂 Parquets détectés :")
print(f"   speech_clean  ({len(clean_parquets)} fichiers) :")
for p in clean_parquets:
    print(f"     {p.split('/')[-1]}  ({_os.path.getsize(p)//1_000_000} MB)")
print(f"   speech_noisy  ({len(noise_parquets)} fichiers) :")
for p in noise_parquets:
    print(f"     {p.split('/')[-1]}  ({_os.path.getsize(p)//1_000_000} MB)")

if not noise_parquets:
    raise FileNotFoundError("Aucun parquet noise trouvé dans /kaggle/input/. Vérifie le dataset dataset_noise.")

# ── 2. Charger et fusionner les parquets noisy ───────────────────────────────
print("\n⏳ Chargement speech_noisy ...")
dfs_noisy = []
for p in noise_parquets:
    part = pd.read_parquet(p)
    dfs_noisy.append(part)
    print(f"   {p.split('/')[-1]} → {len(part)} lignes")

df_noisy = pd.concat(dfs_noisy, ignore_index=True).drop_duplicates(
    subset=["sentence_id","speaker_id"]).reset_index(drop=True)

print(f"\n✅ speech_noisy fusionné : {df_noisy['gender'].value_counts().get('male',0)} hommes, "
      f"{df_noisy['gender'].value_counts().get('female',0)} femmes — {len(df_noisy)} exemples")
print(f"   Durée totale : {df_noisy['duration'].sum()/60:.1f} min")

# ── 3. Fonctions audio (même que cellule 5) ──────────────────────────────────
def resample_to_16k(arr, orig_sr):
    if orig_sr == TARGET_SR:
        return arr
    import torchaudio
    w = torchaudio.functional.resample(
        torch.tensor(arr, dtype=torch.float32).unsqueeze(0),
        orig_freq=orig_sr, new_freq=TARGET_SR)
    return w.squeeze(0).numpy()

def extract_audio(audio_field):
    if isinstance(audio_field, dict):
        raw = audio_field.get('bytes') or audio_field.get('array')
        if isinstance(raw, bytes):
            arr, sr = sf.read(io.BytesIO(raw))
            return arr.astype(np.float32), sr
        if hasattr(raw, '__len__'):
            return np.array(raw, dtype=np.float32), audio_field.get('sampling_rate', 48_000)
    if isinstance(audio_field, bytes):
        arr, sr = sf.read(io.BytesIO(audio_field))
        return arr.astype(np.float32), sr
    raise ValueError(f"Format audio non reconnu : {type(audio_field)}")

def per_score(ref, hyp):
    ref_t = list(ref.replace(' ','_'))
    hyp_t = list(hyp.replace(' ','_'))
    out = jiwer.process_words(' '.join(ref_t), ' '.join(hyp_t))
    return out.wer

# ── 4. Inférence sur speech_noisy avec checkpoint ────────────────────────────
CHECKPOINT_NOISY = "/kaggle/working/results_noisy_checkpoint.json"
results_noisy = []
start_idx = 0

if _os.path.exists(CHECKPOINT_NOISY):
    with open(CHECKPOINT_NOISY) as f:
        saved = _json.load(f)
    if len(saved) < len(df_noisy):
        results_noisy = saved
        start_idx = len(saved)
        print(f"\n♻️  Checkpoint noisy : {start_idx}/{len(df_noisy)} déjà traités, reprise...")
    else:
        results_noisy = saved
        start_idx = len(saved)
        print(f"\n♻️  Checkpoint noisy complet ({len(saved)} ex.) — rechargement direct")

total_audio_s = sum(r["duration"] for r in results_noisy)
total_infer_s = sum(r["inference_time"] for r in results_noisy)

if start_idx < len(df_noisy):
    print(f"\n🚀 Inférence speech_noisy — {len(df_noisy) - start_idx} exemples restants...\n")
    for idx, row in tqdm(df_noisy.iterrows(), total=len(df_noisy), desc="Noisy"):
        if idx < start_idx:
            continue
        ref_ipa  = str(row["phonemized_text"]).strip()
        duration = float(row["duration"])
        try:
            audio_arr, orig_sr = extract_audio(row[SENSOR])
            audio_16k = resample_to_16k(audio_arr, orig_sr)
        except Exception as e:
            print(f"⚠️  Erreur audio idx={idx}: {e}")
            continue
        t0 = time.time()
        with torch.inference_mode():
            pred = asr_pipe({"array": audio_16k, "sampling_rate": TARGET_SR})
        t_inf = time.time() - t0
        pred_ipa = pred["text"].strip()
        total_audio_s += duration
        total_infer_s += t_inf
        results_noisy.append({
            "idx": int(idx), "raw_text": str(row["raw_text"]),
            "ref_ipa": ref_ipa, "pred_ipa": pred_ipa,
            "gender": str(row["gender"]), "speaker_id": str(row["speaker_id"]),
            "duration": duration, "inference_time": t_inf,
            "per": per_score(ref_ipa, pred_ipa),
        })
        if len(results_noisy) % 50 == 0:
            with open(CHECKPOINT_NOISY, "w") as f:
                _json.dump(results_noisy, f)
    with open(CHECKPOINT_NOISY, "w") as f:
        _json.dump(results_noisy, f)

rtf_noisy = total_infer_s / max(total_audio_s, 1)
print(f"\n✅ {len(results_noisy)} inférences noisy terminées  |  RTF: {rtf_noisy:.4f}  ({1/max(rtf_noisy,0.001):.1f}x temps réel)")

# ── 5. Calcul PER pour results (clean) si la colonne per est absente ─────────
for r in results:
    if "per" not in r:
        r["per"] = per_score(r["ref_ipa"], r["pred_ipa"])

# ── 6. Tableau de comparaison Clean vs Noisy ─────────────────────────────────
def stats(res_list, label):
    pers = [r["per"]*100 for r in res_list]
    male   = [r["per"]*100 for r in res_list if r["gender"] == "male"]
    female = [r["per"]*100 for r in res_list if r["gender"] == "female"]
    total_ph = sum(
        len(jiwer.process_words(
            ' '.join(list(r["ref_ipa"].replace(' ','_'))),
            ' '.join(list(r["pred_ipa"].replace(' ','_')))
        ).alignments[0]) for r in res_list[:50]  # approx sur 50 ex
    )
    n_err = sum(
        sum(1 for c in jiwer.process_words(
            ' '.join(list(r["ref_ipa"].replace(' ','_'))),
            ' '.join(list(r["pred_ipa"].replace(' ','_')))
        ).alignments[0] if c.type != 'equal') for r in res_list[:50]
    )
    return {
        "label": label, "n": len(res_list),
        "per_global": np.mean([r["per"] for r in res_list])*100,
        "per_median": np.median(pers),
        "per_std": np.std(pers),
        "per_max": max(pers),
        "per_male": np.mean(male) if male else 0,
        "per_female": np.mean(female) if female else 0,
        "pct_parfait": sum(1 for p in pers if p == 0)/len(pers)*100,
    }

s_clean = stats(results,       "speech_clean")
s_noisy = stats(results_noisy, "speech_noisy")

sep = "-" * 60
print(f"\n{'='*60}")
print(f"  COMPARAISON : speech_clean  vs  speech_noisy")
print(f"  Modèle : {MODEL_NAME}")
print(f"  Capteur : {SENSOR}")
print(f"{'='*60}")
print(f"  {'Métrique':<28} {'Clean':>10} {'Noisy':>10}  {'Δ':>8}")
print(sep)
for key, label in [("n","Exemples"),("per_global","PER global (%)"),
                   ("per_median","PER médian (%)"),("per_max","PER max (%)"),
                   ("per_male","PER hommes (%)"),("per_female","PER femmes (%)"),
                   ("pct_parfait","Phrases parfaites (%)")]:
    c = s_clean[key]
    n = s_noisy[key]
    delta = f"+{n-c:.2f}" if isinstance(c,float) else ""
    vc = f"{c:.2f}" if isinstance(c,float) else str(c)
    vn = f"{n:.2f}" if isinstance(n,float) else str(n)
    print(f"  {label:<28} {vc:>10} {vn:>10}  {delta:>8}")
print(sep)

ratio = s_noisy["per_global"] / max(s_clean["per_global"], 0.01)
print(f"\n  🔴 Dégradation PER : x{ratio:.1f}  ({s_clean['per_global']:.2f}% → {s_noisy['per_global']:.2f}%)")
print(f"  📊 Phrases parfaites : {s_clean['pct_parfait']:.0f}% (clean) → {s_noisy['pct_parfait']:.0f}% (noisy)")
print(f"{'='*60}")

# ── 7. Pires exemples noisy ───────────────────────────────────────────────────
df_noisy_res = pd.DataFrame(results_noisy).sort_values("per", ascending=False)
print(f"\n❌ TOP 5 pires prédictions (speech_noisy) :")
print("="*90)
for _, row in df_noisy_res.head(5).iterrows():
    print(f"  PER: {row['per']*100:.1f}%  |  Genre: {row['gender']}  |  Durée: {row['duration']:.2f}s")
    print(f"  📝 Texte  : {row['raw_text']}")
    print(f"  🔵 Réf IPA: {row['ref_ipa']}")
    print(f"  🔴 Préd   : {row['pred_ipa']}")
    print()

# ── Test statistique : clean vs noisy (C4) ────────────────────────────────
from scipy import stats as _stats

per_clean_arr = np.array([r.get("per", 0) for r in results])
per_noisy_arr = np.array([r.get("per", 0) for r in results_noisy])

stat_mw, pval_mw = _stats.mannwhitneyu(per_clean_arr, per_noisy_arr, alternative="two-sided")
sig_mw = "✅ significatif (p<0.05)" if pval_mw < 0.05 else "⚠️  non significatif"
print(f"\n  Test Mann-Whitney clean vs noisy :")
print(f"  p-value = {pval_mw:.6f} → {sig_mw}")
print(f"  {'La dégradation en conditions bruitées est statistiquement réelle.' if pval_mw < 0.05 else 'Attention : la différence clean/noisy pourrait être due au hasard.'}")

# IC 95% bootstrap sur PER noisy
rng2 = np.random.default_rng(42)
boot_noisy = [np.mean(rng2.choice(per_noisy_arr, size=len(per_noisy_arr), replace=True))*100
              for _ in range(2000)]
ci_lo_n = np.percentile(boot_noisy, 2.5)
ci_hi_n = np.percentile(boot_noisy, 97.5)
print(f"\n  IC 95% PER noisy (bootstrap) : [{ci_lo_n:.3f}% — {ci_hi_n:.3f}%]")


## 16. Correctifs post-traitement — mesure du gain PER réel

Application des règles de correction sur les prédictions existantes :
- `/ɲ/` systématique : `nj → ɲ`
- Phonèmes anglais : `/ɹ/→/ʁ/`, `/θ/→/t/`, `/ɒ/→/ɔ/`, `/aɪ/→/ɛ/`, `/ɪ/→/i/`
- Comparaison PER avant/après sans ré-inférence.

In [ ]:
import re, numpy as np
import jiwer

# ── Règles de correction IPA ─────────────────────────────────────────────────
def fix_ipa(ipa: str) -> str:
    """Applique les correctifs post-traitement sur une séquence IPA prédite."""
    # 1. /ɲ/ systématique : nj → ɲ  (erreur la plus fréquente)
    ipa = re.sub(r'nj', 'ɲ', ipa)
    # 2. Phonèmes anglais → équivalents français
    ipa = re.sub(r'ɹ', 'ʁ', ipa)   # r roulé anglais → r français
    ipa = re.sub(r'θ', 't', ipa)    # th anglais → t
    ipa = re.sub(r'ɒ', 'ɔ', ipa)   # o arrière anglais → o ouvert fr
    ipa = re.sub(r'aɪ', 'ɛ', ipa)  # diphtongue anglaise → è
    ipa = re.sub(r'ɪ', 'i', ipa)   # i bref anglais → i
    return ipa

def per_score(ref: str, hyp: str) -> float:
    rt = list(ref.replace(' ', '_'))
    ht = list(hyp.replace(' ', '_'))
    return jiwer.process_words(' '.join(rt), ' '.join(ht)).wer

# ── Appliquer les correctifs sur tous les résultats clean ────────────────────
print("🔧 Application des correctifs post-traitement...")
print(f"   Règles : nj→ɲ  |  ɹ→ʁ  |  θ→t  |  ɒ→ɔ  |  aɪ→ɛ  |  ɪ→i\n")

corrected = []
for r in results:
    pred_fixed = fix_ipa(r["pred_ipa"])
    per_before = r.get("per", per_score(r["ref_ipa"], r["pred_ipa"]))
    per_after  = per_score(r["ref_ipa"], pred_fixed)
    corrected.append({
        **r,
        "pred_ipa_fixed": pred_fixed,
        "per_before":     per_before,
        "per_after":      per_after,
        "per_gain":       per_before - per_after,
        "changed":        pred_fixed != r["pred_ipa"],
    })

# ── Statistiques globales ─────────────────────────────────────────────────────
n_changed   = sum(1 for r in corrected if r["changed"])
per_before  = np.mean([r["per_before"] for r in corrected]) * 100
per_after   = np.mean([r["per_after"]  for r in corrected]) * 100
gain_abs    = per_before - per_after
gain_pct    = gain_abs / max(per_before, 0.001) * 100

sep = "-" * 60
print(f"{'='*60}")
print(f"  RÉSULTATS — Correctifs post-traitement (speech_clean)")
print(f"  {len(corrected)} exemples analysés")
print(sep)
print(f"  PER avant correctifs   : {per_before:.3f}%")
print(f"  PER après correctifs   : {per_after:.3f}%")
print(f"  Gain absolu            : -{gain_abs:.3f} pts")
print(f"  Gain relatif           : -{gain_pct:.1f}%")
print(f"  Phrases modifiées      : {n_changed} / {len(corrected)} ({n_changed/len(corrected)*100:.1f}%)")
print(sep)

# Gain par règle
for rule_name, pattern, repl in [
    ("nj → ɲ",  r"nj",  "ɲ"),
    ("ɹ → ʁ",   r"ɹ",   "ʁ"),
    ("θ → t",   r"θ",   "t"),
    ("ɒ → ɔ",   r"ɒ",   "ɔ"),
    ("aɪ → ɛ",  r"aɪ",  "ɛ"),
    ("ɪ → i",   r"ɪ",   "i"),
]:
    n_hit = sum(1 for r in corrected if re.search(pattern, r["pred_ipa"]))
    print(f"  {rule_name:<12} → {n_hit:>3} prédictions affectées")

print(f"{'='*60}")

# ── Exemples de corrections ───────────────────────────────────────────────────
improved = sorted([r for r in corrected if r["per_gain"] > 0], key=lambda r: -r["per_gain"])
print(f"\n✅ TOP 5 — Meilleures corrections :")
print("="*90)
for r in improved[:5]:
    g = r["per_gain"]*100
    print(f"  Gain: +{g:.1f}pts  |  {r['raw_text'][:60]}")
    print(f"  Réf    : {r['ref_ipa'][:70]}")
    print(f"  Avant  : {r['pred_ipa'][:70]}")
    print(f"  Après  : {r['pred_ipa_fixed'][:70]}")
    print()

# Cas non améliorés (régression)
regressed = [r for r in corrected if r["per_gain"] < 0]
if regressed:
    print(f"⚠️  Régressions : {len(regressed)} phrases empirées (règles trop agressives)")
    for r in regressed[:3]:
        print(f"   {r['raw_text'][:60]} | {r['per_before']*100:.1f}% → {r['per_after']*100:.1f}%")
else:
    print(f"✅ Aucune régression — les règles sont sûres")

# ── Idem sur noisy si disponible ──────────────────────────────────────────────
if 'results_noisy' in dir() and results_noisy:
    corrected_n = []
    for r in results_noisy:
        pred_fixed = fix_ipa(r["pred_ipa"])
        pb = r.get("per", per_score(r["ref_ipa"], r["pred_ipa"]))
        pa = per_score(r["ref_ipa"], pred_fixed)
        corrected_n.append({**r, "per_before": pb, "per_after": pa, "per_gain": pb-pa})

    pb_n = np.mean([r["per_before"] for r in corrected_n])*100
    pa_n = np.mean([r["per_after"]  for r in corrected_n])*100
    print(f"\n📊 speech_noisy après correctifs :")
    print(f"   PER avant : {pb_n:.3f}%  →  PER après : {pa_n:.3f}%  (gain -{pb_n-pa_n:.3f} pts)")
    print(f"   Dégradation clean→noisy restante : ×{pa_n/max(per_after,0.001):.1f}")

# ── FIX M1 : mettre à jour results avec les prédictions corrigées ────────────
# Les cellules suivantes (eval-audio) utiliseront maintenant pred_ipa corrigé
for r, c in zip(results, corrected):
    r["pred_ipa_raw"]   = r["pred_ipa"]       # sauvegarder l'original
    r["pred_ipa"]       = c["pred_ipa_fixed"]  # écraser avec la version corrigée
    r["per"]            = c["per_after"]       # recalculer le PER

print(f"\n✅ results mis à jour avec prédictions corrigées ({n_changed} phrases modifiées)")
print(f"   pred_ipa_raw : prédiction brute du modèle (conservée)")
print(f"   pred_ipa     : prédiction après correctifs nj→ɲ, anglais→fr")


## 17. Évaluation audio — Prédiction vs Vérité terrain

Pour chaque phrase : audio jouable, **vérité terrain IPA** (Vibravox) vs **prédiction modèle** alignées phonème par phonème.

- 🟢 Correct  🔴 Faux / supprimé  🟠 Ajouté en trop  🟣 Diff. référence  🟡 Phonème anglais hors-vocab FR

Modifier `N_DISPLAY` et `SORT_BY` en haut de la cellule pour personnaliser.

In [ ]:
import base64, io, collections
import soundfile as sf
import numpy as np
from IPython.display import HTML, display

N_DISPLAY = 60
SORT_BY   = "per_desc"

IPA_GUIDE = {
    'a':'a ouvert (patte)','e':'e fermé (été)','ɛ':'e ouvert (fête)',
    'i':'i (vie)','o':'o (mot)','ɔ':'o ouvert (or)','u':'ou (loup)',
    'y':'u (lune)','ø':'eu (feu)','œ':'eu ouvert (peur)','ə':'e muet',
    'ɑ':'a post. (pâte)','ɑ̃':'an nasal','ɛ̃':'in nasal',
    'ɔ̃':'on nasal','œ̃':'un nasal','p':'p','b':'b','t':'t','d':'d',
    'k':'k','ɡ':'g','f':'f','v':'v','s':'s','z':'z liaison',
    'ʃ':'ch','ʒ':'j','m':'m','n':'n','ɲ':'gn palatal (agneau)',
    'ŋ':'ng','l':'l','ʁ':'r uvulaire','j':'y semi-voy.',
    'w':'w labiovél.','_':'frontière mot','∅':'phonème supprimé',
    'ɹ':'r anglais','θ':'th anglais','ɒ':'o anglais',
}
EN_PHONES = {'ɹ','θ','ɒ','ɪ','æ','ð','ʌ'}

KNOWN_LIMITS = {
    'ɲ': ('/ɲ/→/nj/ systématique','100% des cas — soigner, campagne, agneau, règnes'),
    'z': ('/z/ liaison instable', 'Taux 2.7% — vous avez, ils ont, les enfants'),
    'w': ('/w/ sous-représenté',  'Seulement 39 occ. — confondu avec /u/'),
    'j': ('/j/ parfois manqué',   'Position faible ou après consonne'),
    'ə': ('Schwa /ə/ instable',   'Insertion/suppression selon débit'),
}

phone_total  = collections.Counter()
phone_errors = collections.Counter()
for r in results:
    ref_t = list(r['ref_ipa'].replace(' ','_'))
    hyp_t = list(r['pred_ipa'].replace(' ','_'))
    try:
        out = jiwer.process_words(' '.join(ref_t), ' '.join(hyp_t))
        for chunk in out.alignments[0]:
            rs = ref_t[chunk.ref_start_idx:chunk.ref_end_idx]
            for ph in rs:
                if ph != '_': phone_total[ph] += 1
            if chunk.type != 'equal':
                for ph in rs:
                    if ph != '_': phone_errors[ph] += 1
    except:
        pass

difficult = dict(sorted(
    {ph: phone_errors[ph]/phone_total[ph]
     for ph in phone_total if phone_total[ph]>=5 and phone_errors[ph]>0}.items(),
    key=lambda x:-x[1]
))

print(f"Phonemes avec erreurs ({len(results)} phrases):")
for ph, rate in list(difficult.items())[:15]:
    print(f"  /{ph}/ {rate*100:5.1f}%  {'|'*int(rate*30)}  ({phone_errors[ph]}/{phone_total[ph]})")

def audio_b64(audio_field):
    try:
        if isinstance(audio_field, dict):
            raw = audio_field.get('bytes')
            if isinstance(raw, bytes): arr, sr = sf.read(io.BytesIO(raw))
            else: arr = audio_field.get('array',[]); sr = audio_field.get('sampling_rate',48000)
        elif isinstance(audio_field, bytes): arr, sr = sf.read(io.BytesIO(audio_field))
        else: return None
        import torchaudio, torch
        arr_t = torchaudio.functional.resample(
            torch.tensor(arr,dtype=torch.float32).unsqueeze(0),
            orig_freq=sr, new_freq=16000).squeeze(0).numpy()
        buf = io.BytesIO(); sf.write(buf, arr_t, 16000, format='WAV')
        buf.seek(0); return base64.b64encode(buf.read()).decode()
    except: return None

def align_phones(ref_ipa, pred_ipa):
    ref_t = list(ref_ipa.replace(' ','_'))
    hyp_t = list(pred_ipa.replace(' ','_'))
    try: out = jiwer.process_words(' '.join(ref_t), ' '.join(hyp_t))
    except: return [(r,r,'ok') for r in ref_t]
    al = []
    for c in out.alignments[0]:
        rs = ref_t[c.ref_start_idx:c.ref_end_idx]
        hs = hyp_t[c.hyp_start_idx:c.hyp_end_idx]
        if c.type=='equal':
            for r,h in zip(rs,hs): al.append((r,h,'ok'))
        elif c.type=='substitution':
            for i in range(max(len(rs),len(hs))):
                al.append((rs[i] if i<len(rs) else '',hs[i] if i<len(hs) else '','sub'))
        elif c.type=='delete':
            for r in rs: al.append((r,'∅','del'))
        elif c.type=='insert':
            for h in hs: al.append(('∅',h,'ins'))
    return al

def ph_cell(ph, row, status):
    if ph=='_': return '<td style="padding:1px 2px;text-align:center;font-size:11px;color:#444;vertical-align:middle;">|</td>'
    tip   = IPA_GUIDE.get(ph,'/'+ph+'/')
    is_en = ph in EN_PHONES
    C = {
        'ok':  {'ref':('#0d2b1a','#4ade80'),'pred':('#0d2b1a','#4ade80')},
        'sub': {'ref':('#1a1020','#c084fc'),'pred':('#2d1212','#f87171')},
        'del': {'ref':('#1a1020','#c084fc'),'pred':('#2d1212','#f87171')},
        'ins': {'ref':('#0d1f38','#60a5fa'),'pred':('#2d1a08','#fb923c')},
    }
    bg,fg = C.get(status,{'ref':('#1e1e1e','#aaa'),'pred':('#1e1e1e','#aaa')})[row]
    if is_en and row=='pred': bg,fg='#2d1a00','#fde68a'
    bdr = '2px solid #fde68a' if is_en else 'none'
    st = (f'background:{bg};color:{fg};border:{bdr};border-radius:4px;'
          f'padding:3px 5px;font-family:monospace;font-size:13px;font-weight:700;'
          f'text-align:center;min-width:20px;cursor:default;vertical-align:middle;')
    return f'<td style="{st}"><abbr title="{tip}" style="text-decoration:none;">{ph}</abbr></td>'

def ph_table_html(al):
    lbl = ('font-size:9px;color:#7b82a8;text-align:right;padding-right:8px;'
           'white-space:nowrap;vertical-align:middle;min-width:95px;')
    r1 = ''.join(ph_cell(a[0],'ref', a[2]) for a in al)
    r2 = ''.join(ph_cell(a[1],'pred',a[2]) for a in al)
    return (
        '<div style="overflow-x:auto;margin-bottom:4px;">'
        '<table style="border-collapse:separate;border-spacing:2px 2px;">'
        f'<tr><th style="{lbl}">Verite terrain<br>'
        '<span style="font-weight:400;color:#555;">(Vibravox IPA)</span></th>'+r1+'</tr>'
        f'<tr><th style="{lbl}">Predit modele<br>'
        '<span style="font-weight:400;color:#555;">(wav2vec2 CTC)</span></th>'+r2+'</tr>'
        '</table></div>'
    )

def limits_badges(al, ref, pred):
    err_ph = {a[0] for a in al if a[2]!='ok' and a[0] not in ('_','∅','')}
    en_ph  = [a[1] for a in al if a[1] in EN_PHONES and a[2]!='ok']
    nj     = 'ɲ' in ref and 'nj' in pred.replace(' ','')
    bound  = abs(len(ref.split())-len(pred.split()))>=2
    badges = []
    if nj:      badges.append('<span style="background:#2d1212;color:#f87171;font-size:10px;padding:2px 8px;border-radius:12px;border:1px solid #f87171;">RED /nj/ pour /n/</span>')
    if en_ph:   badges.append(f'<span style="background:#2d1a00;color:#fde68a;font-size:10px;padding:2px 8px;border-radius:12px;border:1px solid #fde68a;">JAUNE anglais:/{"/".join(sorted(set(en_ph)))}/</span>')
    if bound:   badges.append('<span style="background:#0d1f38;color:#60a5fa;font-size:10px;padding:2px 8px;border-radius:12px;border:1px solid #60a5fa;">BLEU frontieres</span>')
    for ph in ['z','w','j','ə']:
        if ph in err_ph:
            badges.append(f'<span style="background:#2d1a08;color:#fb923c;font-size:10px;padding:2px 8px;border-radius:12px;border:1px solid #fb923c;">ORANGE /{ph}/ instable</span>')
    if not badges: return '<span style="background:#0d2b1a;color:#4ade80;font-size:10px;padding:2px 8px;border-radius:12px;border:1px solid #4ade80;">OK Aucune limite</span>'
    return ' '.join(badges)

def per_badge(p):
    if p==0:       bg,fg,bd='#0d2b1a','#4ade80','#4ade80'
    elif p<5:      bg,fg,bd='#1a1f2d','#818cf8','#818cf8'
    elif p<15:     bg,fg,bd='#2d1a08','#fb923c','#fb923c'
    else:          bg,fg,bd='#2d1212','#f87171','#f87171'
    return (f'<span style="font-family:monospace;font-size:11px;font-weight:700;'
            f'padding:2px 10px;border-radius:12px;background:{bg};color:{fg};'
            f'border:1px solid {bd};">{p:.1f}%</span>')

if SORT_BY=='per_desc': src=sorted(results,key=lambda r:-r.get('per',0))
elif SORT_BY=='per_asc': src=sorted(results,key=lambda r:r.get('per',0))
else: src=list(results)
src=src[:N_DISPLAY]

cards = ''
for i,r in enumerate(src):
    per  = r.get('per',0)*100
    ref  = r.get('ref_ipa','')
    pred = r.get('pred_ipa','')
    txt  = r.get('raw_text','')
    g    = r.get('gender','')
    dur  = r.get('duration',0)
    idx  = r.get('idx',0)
    gc   = '#60a5fa' if g=='male' else '#f472b6'
    gi   = 'M' if g=='male' else 'F'
    b64  = audio_b64(df_raw.iloc[idx][SENSOR]) if idx<len(df_raw) else None
    au   = f'<audio id="au{i}" src="data:audio/wav;base64,{b64}" preload="none"></audio>' if b64 else ''
    btn  = (f'<button onclick="tp({i})" id="btn{i}" '
            f'style="background:#7c3aed;border:none;border-radius:50%;color:#fff;'
            f'cursor:pointer;font-size:11px;height:26px;width:26px;flex-shrink:0;">P</button>'
            if b64 else '')
    al   = align_phones(ref,pred)
    n_ok = sum(1 for a in al if a[2]=='ok' and a[0]!='_')
    n_er = sum(1 for a in al if a[2]!='ok' and a[0]!='_')
    err_set = sorted({a[0] for a in al if a[2]!='ok' and a[0] not in ('_','∅','')})
    ph_det  = '  .  '.join(f'/{ph}/({difficult.get(ph,0)*100:.1f}%)' for ph in err_set)
    err_span = ('<span style="font-size:10px;color:#7b82a8;">Erreurs : ' + ph_det + '</span>') if ph_det else ''
    cards += (
        f'<div style="background:#1a1d27;border:1px solid #2e3250;border-radius:12px;margin-bottom:14px;overflow:hidden;">'
        f'<div style="display:flex;align-items:center;gap:8px;padding:9px 14px;background:#22263a;border-bottom:1px solid #2e3250;flex-wrap:wrap;">'
        f'<span style="font-family:monospace;font-size:10px;color:#555;">#{i+1}</span>'
        f'<span style="color:{gc};font-size:13px;">{gi}</span>'
        f'<span style="font-family:monospace;font-size:10px;color:#555;">{dur:.1f}s</span>'
        f'<span style="flex:1;font-size:13px;font-weight:600;color:#e2e4f0;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;" title="{txt}">{txt}</span>'
        f'{per_badge(per)}{au}{btn}'
        f'</div>'
        f'<div style="padding:10px 14px 4px;">{ph_table_html(al)}</div>'
        f'<div style="display:flex;gap:14px;padding:3px 14px 4px;flex-wrap:wrap;">'
        f'<span style="font-size:10px;color:#7b82a8;"><b style="color:#e2e4f0;">{n_ok+n_er}</b> phonemes . <b style="color:#4ade80;">{n_ok}</b> corrects . <b style="color:#f87171;">{n_er}</b> erreurs</span>'
        + err_span +
        f'</div>'
        f'<div style="display:flex;gap:6px;padding:2px 14px 10px;flex-wrap:wrap;align-items:center;">'
        f'<span style="font-size:9px;color:#555;white-space:nowrap;">Limites :</span>'
        f'{limits_badges(al,ref,pred)}'
        f'</div></div>'
    )

ph_rows = ''
for ph,rate in list(difficult.items())[:20]:
    bw   = int(rate*180)
    err  = phone_errors[ph]; tot = phone_total[ph]
    desc = IPA_GUIDE.get(ph,'/'+ph+'/')
    note = 'ANGLAIS' if ph in EN_PHONES else ''
    knwn = KNOWN_LIMITS[ph][0] if ph in KNOWN_LIMITS else ''
    if rate>0.02:    sb,sc='#2d1212','#f87171'
    elif rate>0.005: sb,sc='#2d1a08','#fb923c'
    else:            sb,sc='#1a1f2d','#818cf8'
    ph_rows += (
        f'<tr style="border-bottom:1px solid #2e3250;">'
        f'<td style="padding:8px 12px;font-family:monospace;font-size:16px;font-weight:700;color:{sc};background:{sb};border-radius:6px;text-align:center;min-width:40px;">{ph}</td>'
        f'<td style="padding:8px 12px;"><div style="background:#1a2e42;border-radius:3px;height:7px;width:200px;overflow:hidden;"><div style="width:{bw}px;height:100%;background:{sc};border-radius:3px;"></div></div></td>'
        f'<td style="padding:8px 6px;font-family:monospace;font-weight:700;color:{sc};font-size:12px;">{rate*100:.1f}%</td>'
        f'<td style="padding:8px 6px;font-size:11px;color:#7b82a8;">{err}/{tot}</td>'
        f'<td style="padding:8px 6px;font-size:11px;color:#e2e4f0;">{desc}</td>'
        f'<td style="padding:8px 6px;font-size:10px;color:#fde68a;">{note}</td>'
        f'<td style="padding:8px 6px;font-size:10px;color:#7b82a8;">{knwn}</td>'
        f'</tr>'
    )

lim_html = ''
for ph,(lbl,desc) in KNOWN_LIMITS.items():
    rate=difficult.get(ph,0); err=phone_errors.get(ph,0); tot=phone_total.get(ph,0)
    lim_html += (
        f'<div style="background:#1a1d27;border:1px solid #2e3250;border-radius:8px;padding:10px 14px;margin-bottom:8px;">'
        f'<div style="display:flex;align-items:center;gap:8px;margin-bottom:4px;">'
        f'<span style="font-family:monospace;font-size:15px;font-weight:700;color:#f87171;background:#2d1212;padding:2px 8px;border-radius:4px;">{ph}</span>'
        f'<span style="font-size:12px;font-weight:600;color:#e2e4f0;">{lbl}</span>'
        f'<span style="font-family:monospace;font-size:10px;color:#7b82a8;margin-left:auto;">{rate*100:.1f}% ({err}/{tot})</span>'
        f'</div><div style="font-size:11px;color:#7b82a8;">{desc}</div></div>'
    )
lim_html += (
    '<div style="background:#1a1d27;border:1px solid #2e3250;border-radius:8px;padding:10px 14px;margin-bottom:8px;">'
    '<div style="font-size:12px;font-weight:600;color:#fde68a;margin-bottom:4px;">Hallucinations phonemes anglais</div>'
    '<div style="font-size:11px;color:#7b82a8;">Sur mots rares/etrangers : /r/ /th/ /o_ang/. '
    'Ex: logarithme, Leipzig, rollers. Tokens en jaune dans les cartes.</div></div>'
    '<div style="background:#1a1d27;border:1px solid #2e3250;border-radius:8px;padding:10px 14px;">'
    '<div style="font-size:12px;font-weight:600;color:#60a5fa;margin-bottom:4px;">Biais circulaire eSpeak</div>'
    '<div style="font-size:11px;color:#7b82a8;">References IPA generees par eSpeak = meme outil qu\'a l\'entrainement. '
    'PER 0.84% = condition optimale. Sur annotations humaines : 3-15x plus eleve.</div></div>'
)

avg_per = np.mean([r.get('per',0) for r in results])*100
n_perf  = sum(1 for r in results if r.get('per',0)==0)

html = (
    '<style>abbr[title]:hover{text-decoration:underline dotted;}</style>'
    '<div style="background:#0f1117;color:#e2e4f0;padding:18px;border-radius:14px;max-width:1400px;font-family:sans-serif;">'
    '<div style="background:#1a1d27;border:1px solid #2e3250;border-radius:12px;padding:18px;margin-bottom:20px;">'
    '<h2 style="color:#818cf8;font-size:17px;margin:0 0 12px;">Evaluation audio - Prediction vs Verite terrain</h2>'
    '<div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:14px;">'
    f'<div style="background:#22263a;border:1px solid #2e3250;border-radius:8px;padding:10px 14px;"><div style="font-size:22px;font-weight:800;font-family:monospace;color:#4ade80;">{avg_per:.2f}%</div><div style="font-size:10px;color:#7b82a8;">PER moyen</div></div>'
    f'<div style="background:#22263a;border:1px solid #2e3250;border-radius:8px;padding:10px 14px;"><div style="font-size:22px;font-weight:800;font-family:monospace;color:#4ade80;">{n_perf}/{len(results)}</div><div style="font-size:10px;color:#7b82a8;">Phrases parfaites</div></div>'
    f'<div style="background:#22263a;border:1px solid #2e3250;border-radius:8px;padding:10px 14px;"><div style="font-size:22px;font-weight:800;font-family:monospace;color:#818cf8;">{len(difficult)}</div><div style="font-size:10px;color:#7b82a8;">Phonemes avec erreurs</div></div>'
    f'<div style="background:#22263a;border:1px solid #2e3250;border-radius:8px;padding:10px 14px;"><div style="font-size:22px;font-weight:800;font-family:monospace;color:#fb923c;">{N_DISPLAY}</div><div style="font-size:10px;color:#7b82a8;">Phrases affichees</div></div>'
    '</div>'
    '<div style="display:flex;gap:14px;flex-wrap:wrap;font-size:11px;margin-bottom:8px;">'
    '<span style="color:#4ade80;">VERT = Correct</span>'
    '<span style="color:#f87171;">ROUGE = Faux/supprime</span>'
    '<span style="color:#fb923c;">ORANGE = Ajoute</span>'
    '<span style="color:#c084fc;">VIOLET = Diff. reference</span>'
    '<span style="color:#fde68a;">JAUNE = Phoneme anglais hors-FR</span>'
    '</div>'
    '<p style="color:#555;font-size:10px;margin:0;">Survolez un symbole IPA pour la description. P = lire audio. Tri : erreurs en premier.</p>'
    '</div>'
    '<div style="background:#1a1d27;border:1px solid #2e3250;border-radius:12px;padding:16px;margin-bottom:20px;">'
    f'<h3 style="color:#818cf8;font-size:15px;margin:0 0 12px;">Phonemes difficiles - taux erreur sur {len(results)} phrases</h3>'
    '<div style="overflow-x:auto;"><table style="border-collapse:collapse;width:100%;">'
    '<thead><tr style="border-bottom:1px solid #2e3250;">'
    '<th style="padding:5px 12px;text-align:left;font-size:10px;color:#7b82a8;">IPA</th>'
    '<th style="padding:5px 12px;text-align:left;font-size:10px;color:#7b82a8;">Barre</th>'
    '<th style="padding:5px 8px;text-align:center;font-size:10px;color:#7b82a8;">Taux</th>'
    '<th style="padding:5px 8px;text-align:center;font-size:10px;color:#7b82a8;">Err/Total</th>'
    '<th style="padding:5px 8px;text-align:left;font-size:10px;color:#7b82a8;">Description</th>'
    '<th style="padding:5px 8px;text-align:left;font-size:10px;color:#7b82a8;">Note</th>'
    '<th style="padding:5px 8px;text-align:left;font-size:10px;color:#7b82a8;">Limite connue</th>'
    '</tr></thead>'
    f'<tbody>{ph_rows}</tbody></table></div></div>'
    '<div style="background:#1a1d27;border:1px solid #2e3250;border-radius:12px;padding:16px;margin-bottom:20px;">'
    '<h3 style="color:#f87171;font-size:15px;margin:0 0 12px;">Limites identifiees du modele</h3>'
    f'{lim_html}</div>'
    f'{cards}'
    '</div>'
    '<script>'
    'let ca=null,cb=null;'
    'function tp(i){'
    '  const a=document.getElementById("au"+i);'
    '  const b=document.getElementById("btn"+i);'
    '  if(!a)return;'
    '  if(ca&&ca!==a){ca.pause();ca.currentTime=0;if(cb){cb.textContent="P";cb.style.background="#7c3aed";}}'
    '  if(a.paused){a.play();b.textContent="S";b.style.background="#dc2626";'
    '    ca=a;cb=b;a.onended=()=>{b.textContent="P";b.style.background="#7c3aed";}'
    '  }else{a.pause();a.currentTime=0;b.textContent="P";b.style.background="#7c3aed";ca=null;cb=null;}'
    '}'
    '</script>'
)

display(HTML(html))
print(f"\nOK {N_DISPLAY} phrases affichees sur {len(results)} total")
print(f"   Modifier N_DISPLAY et SORT_BY en haut de la cellule pour personnaliser")
